# Fotouni, Cameroon: download streets and assign street ids

Downloads the street network around a point in/near Fotouni, Cameroon,
crops it to the finest OSM boundary containing that point (or a buffer,
see `BUFFER_M` below), and runs `street_ids.label_streets` on it (see
`report/street_identity.md` for what that does and why).

Outputs `nodes.gpkg`, `edges.gpkg`, and `streets.csv` next to this
notebook.


In [1]:
import os

import geopandas as gpd
import osmnx as ox
from ipyleaflet import Map, Marker, basemaps
from shapely.geometry import Point

from osmaddresssystem import osm, street_direction, street_ids, utils


## Hyperparameters

In [2]:
# Used only to seed the initial map center below (its OSM boundary's
# centroid) - the actual download location is whatever the marker on
# the map ends up at, not necessarily this place's own boundary.
TOWN_NAME = "Fotouni, Cameroon"

NETWORK_TYPE = "all"  # "drive", "walk", "bike", "all", ...
BUFFER_M = None  # None -> use the finest OSM boundary containing the point, else buffer (m)

# Streets within this many degrees of each other (undirected orientation,
# folded into 0-90 deg) are considered the same street when crossing an
# intersection with other equal-rank streets.
ANGLE_THRESHOLD_DEG = 20

# Graph nodes closer than this (meters) are treated as one real
# intersection - enables dual-carriageway auto-merging and connector-
# edge bridging. See report/street_identity.md "Node clustering".
NODE_CLUSTER_M = 15.0

# Minimum fuzzy name-similarity score (0-100) for two named edges to
# be treated as the same street - see osmaddresssystem/name_matching.py.
NAME_SIMILARITY_THRESHOLD = 82.0

# Worker processes for the per-node direction-matching step (see
# report/street_identity.md "Performance"). -1 uses all CPUs; only
# actually parallelizes past street_ids._PARALLEL_NODE_THRESHOLD nodes,
# so this is safe to leave on for small downloads too.
N_JOBS = -1

# A street is "transversal" (numbered clockwise around the origin)
# rather than "radial" (numbered outward from it) when the distance
# difference between its two endpoints is less than this percentage of
# its own length - see report/street_identity.md "Street directionality".
TRANSVERSAL_ROAD_PCT = 10.0

TOWN_DIR = os.getcwd()
EXAMPLES_DIR = os.path.dirname(TOWN_DIR)
COUNTRY_STREETS_FOLDER = os.path.join(EXAMPLES_DIR, "streets")
TOWN_STREETS_PATH = os.path.join(TOWN_DIR, "streets.osm")
NODES_GPKG_PATH = os.path.join(TOWN_DIR, "nodes.gpkg")
EDGES_GPKG_PATH = os.path.join(TOWN_DIR, "edges.gpkg")
STREETS_CSV_PATH = os.path.join(TOWN_DIR, "streets.csv")


## Pick a center point

Defaults to the centroid of `TOWN_NAME`'s OSM boundary polygon (the
same "finest boundary containing a point" lookup `osm.download_town_streets`
does internally - see `report/georanks.md`). Drag the marker on the map
to center the download somewhere else instead; the next cell reads
back wherever you left it.


In [3]:
_default_aoi, _default_name = utils.get_town_aoi(TOWN_NAME)
_centroid = _default_aoi.geometry.union_all().centroid
default_location = (_centroid.y, _centroid.x)  # (lat, lon)
print(f"Default center: {default_location} (centroid of {_default_name!r})")

location_map = Map(basemap=basemaps.OpenStreetMap.Mapnik, center=default_location, zoom=13)
location_marker = Marker(location=default_location, draggable=True)
location_map.add(location_marker)
location_map


Found OSM boundary 'Bandja' at level 'town' (12/16) for point (5.32229, 10.22721)
Default center: (5.26531180506471, 10.223308538285993) (centroid of 'Bandja')


Map(center=[5.26531180506471, 10.223308538285993], controls=(ZoomControl(options=['position', 'zoom_in_text', …

In [4]:
# Run this cell after you're happy with the marker's position on the
# map above (or immediately, to keep the default).
LOCATION = tuple(location_marker.location)
print(f"Using LOCATION = {LOCATION}")


Using LOCATION = (5.26531180506471, 10.223308538285993)


## Download and crop the street network

Resolves `LOCATION` to an AOI (the finest OSM administrative/place
boundary containing it, or a buffer), downloads the best-matching
Geofabrik region, and crops it to that AOI.


In [5]:
town_streets_path, aoi, name = osm.download_town_streets(
    LOCATION,
    town_streets_path=TOWN_STREETS_PATH,
    country_streets_folder=COUNTRY_STREETS_FOLDER,
    network_type=NETWORK_TYPE,
    buffer_m=BUFFER_M,
)
print(f"Resolved area: {name!r}")


Found OSM boundary 'Bandja' at level 'town' (12/16) for point (5.26531, 10.22331)
Fetching Geofabrik index from https://download.geofabrik.de/index-v1.json...
Downloaded geofabrik to /home/miguel/Documents/Proyectos/OSMAddressSystem/examples/streets/cameroon.osm.pbf
File '/home/miguel/Documents/Proyectos/OSMAddressSystem/examples/fotouni/streets.osm' already exists. Skipping conversion.
Resolved area: 'Bandja'


## Load the graph and assign street ids

In [6]:
G = ox.graph_from_xml(town_streets_path)
G = ox.project_graph(G, to_crs=aoi.estimate_utm_crs())

nodes_gdf, edges_gdf, streets_df = street_ids.label_streets(
    G,
    angle_threshold_deg=ANGLE_THRESHOLD_DEG,
    n_jobs=N_JOBS,
    node_cluster_m=NODE_CLUSTER_M,
    name_similarity_threshold=NAME_SIMILARITY_THRESHOLD,
)
streets_df


TypeError: label_streets() got an unexpected keyword argument 'n_jobs'

## Give each road a directionality

Now that an origin point is picked (the marker above), classify every
street as radial (numbered outward from the origin) or transversal
(numbered clockwise around it), and record each edge's position along
its street - see `report/street_identity.md` "Street directionality"
for the full rules. `LOCATION` is in lat/lon; it needs to be in the
same projected CRS as `G` (which `label_streets` also required) before
distances/angles from it mean anything.


In [ ]:
origin_point = (
    gpd.GeoSeries([Point(LOCATION[1], LOCATION[0])], crs=4326)
    .to_crs(aoi.estimate_utm_crs())
    .iloc[0]
)
origin = (origin_point.x, origin_point.y)

edges_gdf, streets_df = street_direction.assign_street_directionality(
    nodes_gdf, edges_gdf, streets_df, origin=origin, transversal_road_pct=TRANSVERSAL_ROAD_PCT
)
streets_df


## Save outputs

In [ ]:
nodes_out = street_ids.stringify_nonnumeric_columns(nodes_gdf.reset_index())
edges_out = street_ids.stringify_nonnumeric_columns(edges_gdf.reset_index())
nodes_out.to_file(NODES_GPKG_PATH, driver="GPKG")
edges_out.to_file(EDGES_GPKG_PATH, driver="GPKG")
streets_df.to_csv(STREETS_CSV_PATH, index=False)
print(f"Wrote {NODES_GPKG_PATH}, {EDGES_GPKG_PATH}, {STREETS_CSV_PATH}")


## Next steps

`streets.csv` right now has real OSM `name`s for the streets that
already had one, and `None` for the rest — each of those still needs
its actual name. That's a separate process, not something this
notebook can finish on its own:

1. **Get the official street name list from the town.** Check the data
   license is OSM-compatible (or get written permission) before using
   it for anything beyond this notebook's own reference. See
   `report/upload_street_names_OSM.md` §0 for what to check and record.

2. **Match each official name to a `street_id`.** `streets.csv`'s
   `osmids` column already lists every OSM way under each `street_id`,
   so there's no need to re-derive that mapping from `edges.gpkg` — you
   only need to figure out *which* `street_id` each official name
   refers to (by geometry if the town gave you any, otherwise by eye
   against the plotted network). The skeleton below is what that join
   looks like; it's commented out because there's no real official
   name list for Fotouni yet.

3. **Manually review the match before trusting it.** `label_streets` is
   a heuristic — spot-check a sample of `street_id` groupings against
   aerial imagery or local knowledge, especially for any street you're
   about to bulk-tag.

4. **Upload the confirmed names to OSM.** Follow
   `report/upload_street_names_OSM.md` end to end (licensing check,
   community consultation, dev-API rehearsal, then the real edit via
   iD/JOSM or `osmapi`, scoped to one changeset per town).

5. **Re-run this notebook** against a fresh download once the edits are
   live. Streets you tagged should now come back with real `name`s
   straight from OSM instead of `None`, confirming the round trip.


In [ ]:
# import pandas as pd
#
# MATCHED_NAMES_CSV_PATH = os.path.join(TOWN_DIR, "matched_names.csv")
#
# # official_names.csv: whatever the town provides, reshaped to one row
# # per street_id with the name you've matched to it (see step 2 above).
# official = pd.read_csv(os.path.join(TOWN_DIR, "official_names.csv"))  # columns: street_id, official_name
#
# matched = streets_df.merge(official, on="street_id", how="left")
# matched.to_csv(MATCHED_NAMES_CSV_PATH, index=False)
# matched
